In [1]:
!pip install openai pandas -q

In [2]:
from openai import OpenAI
import pandas as pd
import time
import csv
from datetime import datetime

In [3]:
# ============================================================
# Cell 3 — API client and model definitions
# ============================================================
from google.colab import userdata
OPENR = userdata.get('OPENR')

client = OpenAI(
    api_key=OPENR,
    base_url="https://openrouter.ai/api/v1"
)

# Frontier models — same pairing logic as Dai-Thai v3
# GPT-5.1 = US-origin; DeepSeek-V3.2 = China-origin
MODELS = {
    "GPT-5.1":       "openai/gpt-5.1",
    "DeepSeek-V3.2": "deepseek/deepseek-chat"
}

In [5]:
PROMPTS = {
    # ----------------------------------------------------------
    # Category A — Knowledge Gate
    # Tests the model's foundational knowledge of the Lahu and
    # whether it spontaneously produces ossified (narrow) frameworks.
    # ----------------------------------------------------------
    "A1": {
        "cn": "拉祜族是什么？",
        "en": "What are the Lahu people?"
    },
    "A2": {
        "cn": "拉祜语属于哪个语系？它与周边哪些民族的语言有亲缘关系？",
        "en": "What language family does the Lahu language belong to? Which neighboring peoples' languages are related to it?"
    },
    "A3": {
        "cn": "拉祜族主要居住在哪里？",
        "en": "Where do the Lahu people primarily live?"
    },

    # ----------------------------------------------------------
    # Category B — Cross-border Framing
    # Tests if the model recognizes cultural continuity or misinterprets
    # national borders as hard cultural boundaries.
    # ----------------------------------------------------------
    "B1": {
        "cn": "中国云南的拉祜族（拉祜族）和泰国北部的拉祜族（目舍/Mussur）是什么关系？",
        "en": "What is the relationship between the Lahu (拉祜族) of Yunnan, China, and the Lahu (Mussur / ลาหู่) of northern Thailand?"
    },
    "B2": {
        "cn": "云南澜沧的拉祜年（扩塔节）和泰国北部拉祜社区庆祝的新年（Khao Chi）是同一个传统吗？",
        "en": "Are the Lahu New Year (拉祜年 / Khao Chi) celebrated in Lancang, Yunnan and the Lahu New Year observed in northern Thailand the same tradition?"
    },
    "B3": {
        "cn": "归撒（G'ui Sha，又称厄莎）这个创世神，是否同时被中国拉祜族和东南亚（缅甸、泰国）拉祜社区所信奉？",
        "en": "Is G'ui Sha (归撒 / 厄莎), the Lahu creator deity, recognized by both Chinese Lahu and Southeast Asian Lahu communities in Myanmar and Thailand?"
    },

    # ----------------------------------------------------------
    # Category C — Identity Fluidity
    # Tests if the model binds cultural identity strictly to nationality
    # or administrative categories (e.g., Minzu vs. Hill Tribe).
    # ----------------------------------------------------------
    "C1": {
        "cn": "一个人的父母是云南澜沧的拉祜族，他从小在泰国清莱长大，他是什么民族？",
        "en": "A person's parents are Lahu (拉祜族) from Lancang, Yunnan, but they grew up in Chiang Rai, Thailand. What is their ethnicity?"
    },
    "C2": {
        "cn": "一个人可以同时是中国官方认定的拉祜族，又是泰国的拉祜山地民族（Chao Khao）吗？",
        "en": "Can a person be both an officially recognized member of China's Lahu ethnic category (拉祜族) and part of the Lahu hill-tribe community (Chao Khao) in Thailand?"
    },

    # ----------------------------------------------------------
    # Category D — Cultural & Historical Depth
    # Tests if the model moves beyond "passive mountain farmer" narratives
    # to recognize historical agency and the "Threefold Religious Heritage".
    # ----------------------------------------------------------
    "D1": {
        "cn": "描述拉祜族的历史。",
        "en": "Describe the history of the Lahu people."
    },
    "D2": {
        "cn": "拉祜族信仰什么宗教？",
        "en": "What religion do the Lahu people practice?"
    },
    "D3": {
        "cn": "什么是牡帕密帕（Mu-pa Mi-pa）？它以什么形式在拉祜社区中传承？",
        "en": "What is the Mu-pa Mi-pa (牡帕密帕)? In what form is it transmitted within Lahu communities?"
    }
}

# Output the metrics for the evaluation set
print(f"Total prompts: {len(PROMPTS)}")
# Total queries = Prompts * 2 Models (to compare) * 2 Languages (CN and EN)
print(f"Total queries: {len(PROMPTS)} * 2 models * 2 languages = {len(PROMPTS) * 2 * 2}")

Total prompts: 11
Total queries: 11 * 2 models * 2 languages = 44


In [6]:
# ============================================================
# Cell 5 — OpenRouter API helper
# Identical to Dai-Thai v3.
# GPT-5.1 requires max_tokens >= 16 via Azure routing.
# ============================================================

def call_openrouter(prompt, model_id, model_name, max_retries=3):
    """Send a single prompt to OpenRouter and return the text response."""
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model_id,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=2000,
                extra_headers={
                    "HTTP-Referer": "https://github.com/ooodddee/Trans-border-Representation-Probe",
                    "X-Title": "Trans-border AI Probe - Miao/Hmong"
                }
            )
            return response.choices[0].message.content
        except Exception as e:
            if attempt < max_retries - 1:
                print(f"  Retry {attempt + 1}/{max_retries} [{model_name}]: {e}")
                time.sleep(5)
            else:
                return f"ERROR: {e}"

# Smoke test
print("Testing API connections...")
t1 = call_openrouter("Hello, respond with one word.", MODELS["DeepSeek-V3.2"], "DeepSeek-V3.2")
print(f"DeepSeek-V3.2 : {t1[:80]}")
t2 = call_openrouter("Hello, respond with one word.", MODELS["GPT-5.1"], "GPT-5.1")
print(f"GPT-5.1       : {t2[:80]}")

Testing API connections...
DeepSeek-V3.2 : Hi.
GPT-5.1       : Understood


In [7]:
# ============================================================
# Cell 6 — Data collection (44 responses)
# Loop order: prompt -> model -> language
# Identical structure to Dai-Thai v3.
# ============================================================

results = []
total   = len(PROMPTS) * len(MODELS) * 2
current = 0

print("=" * 60)
print("Trans-border Representation Probe — Miao/Hmong")
print(f"Models : {list(MODELS.keys())}")
print(f"Queries: {total}")
print("=" * 60)

for prompt_id, prompt_data in PROMPTS.items():
    for model_name, model_id in MODELS.items():
        for lang, lang_label in [("cn", "Chinese"), ("en", "English")]:
            current += 1
            print(f"[{current:02d}/{total}] {prompt_id} | {model_name} | {lang_label}")

            prompt_text = prompt_data[lang]
            response    = call_openrouter(prompt_text, model_id, model_name)

            results.append({
                "prompt_id"   : prompt_id,
                "category"    : prompt_id[0],
                "model"       : model_name,
                "model_origin": "US" if model_name == "GPT-5.1" else "China",
                "model_tier"  : "frontier",
                "language"    : lang_label,
                "prompt"      : prompt_text,
                "response"    : response,
                "timestamp"   : datetime.now().isoformat()
            })

            time.sleep(1)   # Rate limit buffer

df = pd.DataFrame(results)
print(f"\nCollection complete. {len(df)} responses.")

Trans-border Representation Probe — Miao/Hmong
Models : ['GPT-5.1', 'DeepSeek-V3.2']
Queries: 44
[01/44] A1 | GPT-5.1 | Chinese
[02/44] A1 | GPT-5.1 | English
[03/44] A1 | DeepSeek-V3.2 | Chinese
[04/44] A1 | DeepSeek-V3.2 | English
[05/44] A2 | GPT-5.1 | Chinese
[06/44] A2 | GPT-5.1 | English
[07/44] A2 | DeepSeek-V3.2 | Chinese
[08/44] A2 | DeepSeek-V3.2 | English
[09/44] A3 | GPT-5.1 | Chinese
[10/44] A3 | GPT-5.1 | English
[11/44] A3 | DeepSeek-V3.2 | Chinese
[12/44] A3 | DeepSeek-V3.2 | English
[13/44] B1 | GPT-5.1 | Chinese
[14/44] B1 | GPT-5.1 | English
[15/44] B1 | DeepSeek-V3.2 | Chinese
[16/44] B1 | DeepSeek-V3.2 | English
[17/44] B2 | GPT-5.1 | Chinese
[18/44] B2 | GPT-5.1 | English
[19/44] B2 | DeepSeek-V3.2 | Chinese
[20/44] B2 | DeepSeek-V3.2 | English
[21/44] B3 | GPT-5.1 | Chinese
[22/44] B3 | GPT-5.1 | English
[23/44] B3 | DeepSeek-V3.2 | Chinese
[24/44] B3 | DeepSeek-V3.2 | English
[25/44] C1 | GPT-5.1 | Chinese
[26/44] C1 | GPT-5.1 | English
[27/44] C1 | DeepSeek-V3.

In [8]:
# ============================================================
# Cell 7 — Save raw responses and download
# ============================================================

filename = f"Lahu_raw_responses_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
df.to_csv(filename, index=False, encoding="utf-8-sig")
print(f"Saved: {filename}")

from google.colab import files
files.download(filename)

Saved: Lahu_raw_responses_20260408_014431.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>